# OCR

Nesse notebook vamos explorar a extração de textos de imagens (OCR) usando a biblioteca `easyocr`.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install easyocr
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que usamos em outros notebooks, vamos utilizar mais algumas.
* `easyocr`: Biblioteca open source para trabalhar com OCR.

In [ ]:
import easyocr

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image

## 1. Verificando se o OCR está OK

Vamos começar com um teste bem simples para saber se o **EasyOCR** está funcionando corretamente.

### 1.1. Carregando imagem

Nesse notebook vamos trabalhar apenas com arquivos de imagens. Em cenários reais, os arquivos provavelmente estarão em formato PDF. Nesse caso, deve haver uma conversão da página em questão em PNG ou em array de bytes antes de processar o OCR.

**Observação:** A biblioteca EasyOCR só trabalha com imagens no formato RGB.

In [ ]:
img_bgr = cv2.imread('imagens/03/teste-ocr.webp')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

### 1.2. Carregando o Leitor OCR

Vamos instanciar um objeto do tipo `easyocr.Reader` para extrair os textos das imagens.

In [ ]:
reader = easyocr.Reader(
    ['pt', 'en'], # Idiomas que queremos processar
    gpu=False
)

### 1.3. Processando o OCR

Agora vem a extração dos textos. 

**Observações:**
- Como estamos lidando com redes neurais, o uso de `CPU` torna o processo bem lento.
- Resolução da imagem também é um fator que implica na performance do OCR.

In [ ]:
resultados = reader.readtext(img_rgb)

Agora vamos entender o que o método `readtext()` retorna.

In [ ]:
print(f"Total de ocorrências: {len(resultados)}")

resultados[0]

In [ ]:
for bbox, texto, confianca in resultados:
    print(f"{texto:40} {confianca:.2%}")

### 1.4. Apresentando Resultados

Vamos ver 3 formas de visualizar os resultados obtidos.

In [ ]:
def apresentar_resultados(img_rgb: np.ndarray, resultados: list, *, mostrar_texto: bool = False) -> None:
    img_draw = img_rgb.copy()

    for bbox, texto, confianca in resultados:

        pontos = np.array(bbox, dtype=np.int32)

        cv2.polylines(
            img_draw,
            [pontos],
            True,
            (0, 255, 0),
            2
        )

        if mostrar_texto:
            x = pontos[0][0]
            y = pontos[0][1] - 10

            cv2.putText(
                img_draw,
                texto,
                (x, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

    plt.figure(figsize=(15, 10))
    plt.imshow(img_draw)
    plt.axis('off')
    plt.show()

#### 1.4.1. Apenas Bounding Boxes

In [ ]:
apresentar_resultados(img_rgb, resultados)

#### 1.4.2. Bounding Boxes com Texto

In [ ]:
apresentar_resultados(img_rgb, resultados, mostrar_texto=True)

#### 1.4.2. Apenas Texto

In [ ]:
def extrair_texto(resultados: list) -> str:
    textos = [texto for _, texto, _ in resultados]
    return '\n'.join(textos)

In [ ]:
texto = extrair_texto(resultados)
print(texto)

## 2. Outro Exemplo

Vamos demonstrar em outra imagem com texto corrido com fontes em diversas cores.

In [ ]:
img_bgr = cv2.imread('imagens/03/teste-ocr-2.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

In [ ]:
resultados = reader.readtext(img_rgb)

In [ ]:
texto = extrair_texto(resultados)
print(texto)

Nada mal!

## 3. Documentos mais realistas

Boa parte dos documentos que processamos no dia-a-dia não são páginas de livros de romance. São bem mais complexos. Possuem tabelas, informações em locais distintos. É aqui que vamos começar a perceber as limitações do OCR.

In [ ]:
img_bgr = cv2.imread('imagens/03/nota-fiscal.png')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

In [ ]:
resultados = reader.readtext(img_rgb)

In [ ]:
texto = extrair_texto(resultados)
print(texto)

Apesar da acurácia de extração ter sido positiva, processar informações dessa forma é extremamente complicado. 

Note que:
- A opção de destinação do documento não foi capturada.
- A data de emissão veio em 3 extrações diferentes.

In [ ]:
apresentar_resultados(img_rgb, resultados, mostrar_texto=True)

## 4. Pré-processamento

A biblioteca EasyOCR já realiza diversos tratamentos antes de processar o OCR. Nem toda biblioteca ou OCR fará isso. Em alguns casos, será necessário realizar alguns ajustes/ calibrações para documentos escaneados.

In [ ]:
img_bgr = cv2.imread('imagens/04/ocr_bad_scan.jpg')
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.show()

In [ ]:
def enhance(img, to_gray=True, normalize=True, sharpen_value=0, thres_radius=5, thres_constant=5, noise_reduction=0):
    """
    Aplica alguns filtros para melhorar scans ruins
    
    :param img: imagem a ser tratada
    :param to_gray: indica se a imagem deve ser convertida para escala de cinza
    :param normalize: indica se a imagem deve ser normalizada
    :param sharpen_value: valor de aumento de nitidez
    :param thres_radius: se maior que 0, aplica binarização adaptiva com o raio indicado
    :param thres_constant: valor de atenuação da binarização
    :param noise_reduction: valor de redução de ruido
    :returns: imagem tratada
    """
    mat = img.copy()

    # Converte em escala de cinza dependendo dos parâmetros
    if to_gray or thres_radius > 0:
        mat = cv2.cvtColor(mat, cv2.COLOR_RGB2GRAY)
    
    # Normaliza imagem
    if normalize:
        cv2.normalize(mat, mat, 0, 255, cv2.NORM_MINMAX)
    
    # Aplica o conceito de unsharp mask
    if sharpen_value > 0:
        block_size = (sharpen_value * 2 + 1)**2
        blurred = cv2.GaussianBlur(mat, (block_size, block_size), 10.0)
        mat = cv2.addWeighted(mat, 1.5, blurred, -0.5, 0, mat)
    
    # Aplica a binarização adaptiva
    if thres_radius > 0:
        mat = cv2.adaptiveThreshold(mat, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 
                                    thres_radius*2+1, thres_constant)
    
    # Remove um pouco do ruído gerado
    if noise_reduction > 0:
        mat = cv2.medianBlur(mat, noise_reduction*2+1)

    # Caso tenha sido convertido para escala de cinza, volta para RGB
    if to_gray or thres_radius > 0:
        mat = cv2.cvtColor(mat, cv2.COLOR_GRAY2RGB)

    return mat

In [ ]:
enhanced_img = enhance(img_rgb, thres_radius=12, thres_constant=12, noise_reduction=1)

plt.figure(figsize=(12, 8))
plt.imshow(enhanced_img)
plt.axis('off')
plt.show()

In [ ]:
# Senta que lá vem história...
resultados = reader.readtext(enhanced_img)

In [ ]:
texto = extrair_texto(resultados)
print(texto)